# Изграждане с моделите от фамилията Meta

## Въведение

Този урок ще обхване:

- Изследване на двата основни модела от фамилията Meta - Llama 3.1 и Llama 3.2, както и наличните към момента алтернативи, използвани в примерите за код (Llama 3.3 и Llama 4)
- Разбиране на случаите на използване и сценариите за всеки модел
- Примерен код, показващ уникалните характеристики на всеки модел


## Фамилията модели Meta

В този урок ще изследваме 2 модела от фамилията Meta или "Стадото на Ллама" - Llama 3.1 и Llama 3.2. Изпълнимите примери за код използват Llama 3.3 и Llama 4, които са налични алтернативи в каталога на Microsoft Foundry Models.

Тези модели идват в различни варианти и са налични в [каталога на Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst).

> **Забележка:** GitHub Models ще бъде закрит в края на юли 2026 г. Тук има повече информация за използването на [Microsoft Foundry Models](https://learn.microsoft.com/azure/ai-foundry/model-inference/overview?WT.mc_id=academic-105485-koreyst) за прототипиране с AI модели.

> **Забележка:** Конкретните версии на моделите, спрямо които този урок беше първоначално написан (Llama 3.1 70B/405B Instruct, Llama 3.2 11B/90B Vision Instruct), вече са прекратени в каталога на Microsoft Foundry Models. По-долу примерите за код използват `Llama-3.3-70B-Instruct` и `Llama-4-Scout-17B-16E-Instruct` като налични към момента алтернативи в каталога на Microsoft Foundry Models, така че те все още работят.

Варианти на моделите (както са разгледани първоначално в този урок):
- Llama 3.1 - 70B Instruct (прекратен)
- Llama 3.1 - 405B Instruct (прекратен)
- Llama 3.2 - 11B Vision Instruct (прекратен)
- Llama 3.2 - 90B Vision Instruct (прекратен)

Налични в момента алтернативи, използвани в примерите за код:
- Llama 3.3 - 70B Instruct (`Llama-3.3-70B-Instruct`)
- Llama 4 Scout - 17B 16E Instruct (`Llama-4-Scout-17B-16E-Instruct`)

*Забележка: Обсъдените по-горе вариации на Llama 3 се запазват за исторически контекст и вече не са налични в Microsoft Foundry Models.*



## Llama 3.1 

С 405 милиарда параметри, Llama 3.1 попада в категорията на отворените източници LLM. 

Този модел е надградена версия на по-ранния Llama 3, предлагайки: 

- По-голям прозорец на контекст - 128k токена срещу 8k токена 
- По-голям максимален брой изходни токени - 4096 срещу 2048 
- По-добра многезвучна поддръжка - благодарение на увеличения брой обучителни токени 

Това позволява на Llama 3.1 да обработва по-сложни случаи при изграждането на GenAI приложения, включително: 
- Родно извикване на функции - възможност за извикване на външни инструменти и функции извън работния поток на LLM
- По-добра производителност при RAG - благодарение на по-големия прозорец на контекст 
- Генериране на синтетични данни - възможност за създаване на ефективни данни за задачи като допълнително обучение 



### Извикване на родни функции 

Llama 3.1 е оптимизиран да бъде по-ефективен при извикване на функции или инструменти. Той също така има два вградени инструмента, които моделът може да разпознае, че трябва да използва въз основа на заявката от потребителя. Тези инструменти са: 

- **Brave Search** - Може да се използва за получаване на актуална информация като времето чрез търсене в мрежата 
- **Wolfram Alpha** - Може да се използва за по-сложни математически изчисления, така че не е необходимо да пишете свои собствени функции. 

Можете също да създавате свои собствени персонализирани инструменти, които LLM може да извиква. 

В примерния код по-долу: 

- Дефинираме наличните инструменти (brave_search, wolfram_alpha) в системната заявка. 
- Изпращаме заявка от потребителя, която пита за времето в определен град. 
- LLM ще отговори с извикване на инструмента Brave Search, което ще изглежда така `<|python_tag|>brave_search.call(query="Stockholm weather")` 

*Забележка: Този пример само извършва извикване на инструмента, ако искате да получите резултатите, ще трябва да си създадете безплатен акаунт на страницата на Brave API и да дефинирате функцията сама` 


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import AssistantMessage, SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
# Meta-Llama-3.1-405B-Instruct was retired from Microsoft Foundry on 2026-06-13 with no replacement listed;
# using Llama-3.3-70B-Instruct, a currently available alternative, instead
model_name = "Llama-3.3-70B-Instruct"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


tool_prompt=f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Environment: ipython
Tools: brave_search, wolfram_alpha
Cutting Knowledge Date: December 2023
Today Date: 23 July 2024

You are a helpful assistant<|eot_id|>
"""

messages = [
    SystemMessage(content=tool_prompt),
    UserMessage(content="What is the weather in Stockholm?"),

]

response = client.complete(messages=messages, model=model_name)

print(response.choices[0].message.content)






### Llama 3.2 

Въпреки че е LLM, едно ограничение на Llama 3.1 е мултимодалността. Това означава възможността за използване на различни типове входни данни като изображения като подсказки и предоставяне на отговори. Тази способност е една от основните характеристики на Llama 3.2. Тези характеристики включват също: 

- Мултимодалност - има способността да оценява както текстови, така и образни подсказки 
- Малки до средни варианти (11B и 90B) - това осигурява гъвкави опции за внедряване, 
- Текстови варианти (1B и 3B) - това позволява моделът да бъде внедрен на крайни / мобилни устройства и осигурява ниска латентност 

Поддръжката на мултимодалност представлява голяма стъпка в света на отворените модели. Примерът с код по-долу използва както изображение, така и текстова подсказка, за да получи анализ на изображението. Тъй като Llama 3.2 90B Vision Instruct е прекратен, примерът използва `Llama-4-Scout-17B-16E-Instruct`, настоящ наличен нативно мултимодален заместител. 

### Поддръжка на мултимодалност с Llama 3.2


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    SystemMessage,
    UserMessage,
    TextContentItem,
    ImageContentItem,
    ImageUrl,
    ImageDetailLevel,
)
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
# Llama-3.2-90B-Vision-Instruct was retired from Microsoft Foundry on 2026-06-13 with no replacement listed;
# using Llama-4-Scout-17B-16E-Instruct, a currently available natively multimodal alternative, instead
model_name = "Llama-4-Scout-17B-16E-Instruct"


client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = client.complete(
    messages=[
        SystemMessage(
            content="You are a helpful assistant that describes images in details."
        ),
        UserMessage(
            content=[
                TextContentItem(text="What's in this image?"),
                ImageContentItem(
                    image_url=ImageUrl.load(
                        image_file="sample.jpg",
                        image_format="jpg",
                        detail=ImageDetailLevel.LOW)
                ),
            ],
        ),
    ],
    model=model_name,
)

print(response.choices[0].message.content)


## Обучението не спира тук, продължете пътешествието

След като завършите този урок, разгледайте нашата [колекция за обучение по Генеративен ИИ](https://aka.ms/genai-collection?WT.mc_id=academic-105485-koreyst), за да продължите да развивате знанията си за Генеративен ИИ!


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводачески услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
